In [13]:
# ! pip install selenium
# ! pip install selenium-stealth
# ! pip install beautifulsoup4
# ! pip install lxml
# ! pip install requests
# ! pip install pandas

In [14]:
from selenium import webdriver
from selenium_stealth import stealth

from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC

from selenium.common.exceptions import TimeoutException
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import StaleElementReferenceException
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
import requests
import csv
import time
from bs4 import BeautifulSoup

In [15]:
# Set up Chrome options
chrome_options = Options()
# chrome_options.add_argument("--headless")  # Run headless Chrome
chrome_options.add_argument("--window-size=1920,1080")  # Set window size
# chrome_options.add_argument('--headless=new') # ensure GUI is off
chrome_options.add_argument('start-maximized') #sets the browser to maximixed view
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument("--disable-search-engine-choice-screen")
chrome_options.add_argument('disable-infobars')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument("--ignore-certificate-errors")
chrome_options.add_argument("--window-size=2560,1440") # set specific window size for the browser,
chrome_options.add_argument("--incognito") # set the browser mode to incognito
chrome_options.add_argument('--enable-javascript')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # selenium stealth option to enable automation
chrome_options.add_experimental_option('useAutomationExtension', False)
chrome_options.add_argument("lang=en")
chrome_options.add_argument("--disable-extensions")
driver = webdriver.Chrome(options=chrome_options)

#if you did not set the chrome driver to you environmental variable, uncomment the line below.
# driver = webdriver.Chrome(options=chrome_options, executable_path=r"path_to_extracted_driver\chromedriver.exe")

url='https://www.momondo.com/hotel-search/Tallinna,Viro-p6073/2024-10-07/2024-10-09/2adults;map?sort=rank_a  '
driver.get(url=url)
driver.implicitly_wait(5)

In [16]:
page_sources=[]
pages=10

try:
    # Wait for the cookie popup and click 'Accept all' or 'Reject all'
    cookie_popup = WebDriverWait(driver, 30).until(
        
        EC.visibility_of_element_located((By.XPATH, "/html/body/div[5]/div/div[2]/div/div/div[3]/div/div[1]/button[1]/div/div"))).click() #MOMONDO


    
except TimeoutException:
    print("Cookie consent popup not found")
    
data=[]
for i in range(1,pages+1):
  
  WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.CLASS_NAME, "S0Ps")))
  properties = driver.find_elements(By.CLASS_NAME, "S0Ps")
  print(f"On page {i}")
  print(f"Found {len(properties)} properties")

  for prop in properties:
    url = prop.find_element(By.CLASS_NAME, "FLpo-hotel-name").find_element(By.TAG_NAME, "a").get_attribute("href")
    price = None
    price_full_div = prop.find_element(By.CLASS_NAME, "zV27-price-section")

    try:
      price_div = price_full_div.find_element(By.CLASS_NAME, "c1XBO")
      price = price_div.text[1:].replace(",", "")
    except:
      pass

    data.append([url, price])
    
  # Click next

try:
    print("Finding next page...")
    previous,next_page= WebDriverWait(driver,10,
                ignored_exceptions=(
                        NoSuchElementException, 
                        StaleElementReferenceException)).until(EC.visibility_of_all_elements_located((By.CLASS_NAME,'A8fY-chevron')))
    next_page.click()
except Exception as e:
    print(f'An exception occurred when clicking next: {e}')
  

On page 1
Found 28 properties
On page 2
Found 28 properties
On page 3
Found 28 properties
On page 4
Found 28 properties
On page 5
Found 28 properties
On page 6
Found 28 properties
On page 7
Found 28 properties
On page 8
Found 28 properties
On page 9
Found 28 properties
On page 10
Found 28 properties
Finding next page...


In [17]:
number_of_nights=3
def process_property(driver, price):
  el = {}

  main_div = driver.find_element(By.CLASS_NAME, "eu4c")

  el["name"] = main_div.find_element(By.CLASS_NAME, "c3xth-hotel-name").text

  try:
    el["stars"] = driver.find_element(By.CLASS_NAME, "c3xth-stars-in-title").find_element(By.TAG_NAME, "span").text.split()[0]
  except:
    el["stars"] = None

  try:
    el["address"] = main_div.find_element(By.CLASS_NAME, 'c3xth-address').text
  except:
    el["address"] = None

  h2_rating = driver.find_element(By.XPATH, '//*[@data-section-name="reviews"]').find_element(By.CLASS_NAME, "T_-S-section-header")
  if h2_rating.text == "Reviews":
    el["rating"] = driver.find_element(By.CLASS_NAME, "TnzK-score").text
    num_of_ratings = driver.find_element(By.CLASS_NAME, "TnzK-count").text
    number = num_of_ratings.split()[0]
    el["number of ratings"] = number
  else:
    el["rating"] = None
    el["number of ratings"] = 0

  # There are some hotels which need to be contacted, and so the regular price divs are missing
  if price:
    el["total price ($)"] = price
    el["price per night ($)"] = float(el["total price ($)"]) / number_of_nights
  else:
    el["total price ($)"] = None
    el["price per night ($)"] = None

  try:
    show_more_button = driver.find_element(By.CLASS_NAME, 'b40a-inline-read-more-button')
    driver.execute_script("arguments[0].click();", show_more_button)
    el["description"] = driver.find_element(By.CLASS_NAME, 'b40a-desc-wrap--full').text
  except:
    try:
      el["description"] = driver.find_element(By.CLASS_NAME, 'b40a-description-simple').text
    except:
      el["description"] = None

  el["images"] = []
  images = driver.find_elements(By.CLASS_NAME, 'f800-image')
  for image in images:
    el["images"].append(image.get_attribute("src"))

  return el

In [18]:
data

[['https://www.momondo.com/hotel-search/Hotel-Bern-by-TallinnHotels,Tallinn-p6073-h166281-details/2024-10-07/2024-10-09/2adults?psid=cKHEN-EAVQ&pm=totaltaxes#overview',
  '95'],
 ['https://www.momondo.com/hotel-search/Park-Inn-by-Radisson-Meriton-ConferenceSpa-Tallin,Tallinn-p6073-h5525-details/2024-10-07/2024-10-09/2adults?psid=cKHEN-EAVQ&pm=totaltaxes#overview',
  '140'],
 ['https://www.momondo.com/hotel-search/Citybox-Tallinn-City-Center,Tallinn-p6073-h5966183-details/2024-10-07/2024-10-09/2adults?psid=cKHEN-EAVQ&pm=totaltaxes#overview',
  '114'],
 ['https://www.momondo.com/hotel-search/Metropol-Spa-Hotel,Tallinn-p6073-h3781324-details/2024-10-07/2024-10-09/2adults?psid=cKHEN-EAVQ&pm=totaltaxes#overview',
  '174'],
 ['https://www.momondo.com/hotel-search/Savoy-Boutique-Hotel-by-TallinnHotels,Tallinn-p6073-h157901-details/2024-10-07/2024-10-09/2adults?psid=cKHEN-EAVQ&pm=totaltaxes#overview',
  '180'],
 ['https://www.momondo.com/hotel-search/Rija-Fonnental-Design-Hotel-Tallinn,Tallinn

In [19]:
properties_scraped = []

for url, price in data:
  driver.get(url)

  success = False
  for attempt in range(3):
    try:
      WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CLASS_NAME, "eu4c")))
      success = True
      break
    except:
      driver.refresh()

  if success:
    el = process_property(driver, price)
  else:
    print(f"Skipping {url}")

  print(el)
  properties_scraped.append(el)

{'name': 'Hotel Bern by TallinnHotels', 'stars': '4', 'address': 'Aia 10, 10111 Tallinn, Harjumaa', 'rating': '8.3', 'number of ratings': '3501', 'total price ($)': '95', 'price per night ($)': 31.666666666666668, 'description': 'Warm welcome, cosy atmosphere and modern accommodation in Tallinn Old Town. Economy Bern Hotel Tallinn provides the best and personal service and is the most helpful host to make your visit a memorable experience. Thanks to the excellent location of the Hotel, you can find innumerable sights, restaurants, boutiques, parlours and shopping centres in the immediate vicinity of the hotel. Tallinn airport is barely a 15 minutes drive from the hotel, and you can reach the passenger port as well railway station in 5 minutes.iHotelier/iStayRead less', 'images': ['https://content.r9cdn.net/rimg/himg/2c/f5/7c/expediav2-166281-14842c-089608.jpg?width=1020&height=1020&xhint=540&yhint=333&crop=true&watermarkheight=28&watermarkpadding=10', 'https://content.r9cdn.net/rimg/hi

In [2]:
import ast

In [3]:
# with open('output.txt', 'r') as f:
#     page_sources=ast.literal_eval(f.read())

In [ ]:
# len(page_sources)

In [9]:
# page1=page_sources[0]

In [8]:
# for page in page_sources:
#     soup = BeautifulSoup(page, 'html.parser')
#     hotels_soup=soup.find_all('li',class_='PropertyCard')
#     name = rating = address = stars = price = "N/A"
#     images = []
#     hotels=[]
#     number_of_nights=3
    

#     for element in hotels_soup:
#         try:
#             # Initialize variables with default values (None) in case extraction fails
#             name = rating = address = text_rating = price = None
#             images = []

#             # Extract the hotel name (set None if it fails)
#             try:
#                 name = element.find('h3').get_text(strip=True)
#             except AttributeError:
#                 name = None

#             # Extract the rating (check if there's at least one <p> tag with that class)
#             try:
#                 p_tags = element.find_all('p', class_='dynamic-style-typographystyle-3')
#                 if len(p_tags) > 0:
#                     rating = p_tags[0].get_text(strip=True)
#                 else:
#                     rating = None
#             except AttributeError:
#                 rating = None

#             # # Extract the text rating (check if there's a second <p> tag)
#             try:
#                 stars = element.find('span',class_='sc-crrsfI')
                
#                 if stars:
#                     stars = stars.get_text(strip=True).split(' ')[0]
#                 else:
#                     stars = None
#             except AttributeError:
#                 price("Error here1")
#                 stars = None

#             # Extract the address (set None if it fails)
#             try:
#                 address = element.find('span', class_='dynamic-style-typographystyle-3').get_text(strip=True).split(" - ")[0]
#             except AttributeError:
#                 address = None

#             # Extract image URLs (set an empty list if it fails)
#             # try:
#             #     image_tags = element.find_all('img', class_='thumbnail-image')
#             #     images = [img['src'] for img in image_tags if 'src' in img.attrs]
#             # except AttributeError:
#             #     images = []

#             # Extract price per night (set None if it fails)
#             try:
#                 price = element.find('span', class_='PropertyCardPrice__Value').get_text(strip=True)
#             except AttributeError:
#                 price = None

#             # Append the extracted data to the hotels list
#             hotels.append({
#                 "name": name,
#                 "rating": rating,
#                 "address": address,
#                 "hotel rating": stars,
#                 # "images": images,  # Uncomment if you need to include the image URLs
#                 "price per night($)": price,
#             })
        
#         except Exception as e:
#             print(f"An error occurred while processing a hotel: {e}")
#             continue


In [7]:
# len(hotels)

24

In [10]:
import pandas as pd

df = pd.DataFrame(properties_scraped)

# # Convert the price column to float

# df['price per night($)'] = df['price per night($)'].str.replace(',', '').astype(float)

# df
# df.dropna(subset=['rating'],inplace=True)


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12 entries, 0 to 23
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   name                12 non-null     object 
 1   rating              12 non-null     object 
 2   address             12 non-null     object 
 3   hotel rating        12 non-null     object 
 4   price per night($)  12 non-null     float64
dtypes: float64(1), object(4)
memory usage: 576.0+ bytes


In [62]:
# df["total price ($)"]=df["price per night($)"]*number_of_nights

In [11]:
df

,name,rating,address,hotel rating,price per night($)
0,Old Town - Dunkri Apartment,9.0,"Tallinn Old Town, Tallinn",4,155.0
1,AirHome - Owl's Nest,9.1,"Kalamaja, Tallinn",4.5,323.0
2,3 room central apartmend 90m2 parking for one car,9.4,Managed by a private host,5,76.0
3,Tallinn Central City apartment,7.9,"Lower Town, Tallinn",tooltip,80.0
9,Paivilla Boutique Hotel,8.0,"Kristiine, Tallinn",tooltip,74.0
11,Saia Forest House,9.6,"Suburbs, Tallinn",tooltip,93.0
12,room to the east,9.0,Managed by a private host,tooltip,55.0
13,Rixwell Viru Square Hotel Tallinn,8.0,"Tallinn Old Town, Tallinn",3,66.0
14,Stereo House by Larsen,9.0,"Nomme, Tallinn",tooltip,84.0
20,Railrooms Traincar Hostel,7.9,"Kalamaja, Tallinn",tooltip,59.0


In [ ]:
# hotels_soup[0].find_all('span',class_='sc-crrsfI')#.get_text(strip=True)